**IMPORTING LIBRARIES**

過濾日期：利用 _TABLE_SUFFIX 只抓取 2020/11/01 到 2020/11/07 的資料。

過濾事件：只保留「進站、看商品、加購物車、結帳、購買」這 5 個漏斗核心事件，大幅減少檔案大小。

攤平 JSON：把原本隱藏在 event_params 裡的 session_id 抽取出來，這樣你之後用 dbt 就能串接同一次瀏覽的漏斗。

In [ ]:
SELECT
  -- 1. 基本時間與事件資訊
  event_date,
  TIMESTAMP_MICROS(event_timestamp) AS event_time,
  event_name,
  
  -- 2. 顧客識別與工作階段 (Session)
  user_pseudo_id,
  (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id') AS session_id,
  
  -- 3. 裝置與地區維度 (滿足轉換率分析)
  device.category AS device_category,
  device.operating_system,
  geo.country,
  geo.city,
  
  -- 4. 流量來源維度 (滿足轉換率分析)
  traffic_source.source AS traffic_source,
  traffic_source.medium AS traffic_medium,
  
  -- 5. 電商交易資料 (滿足高價值顧客分析)
  ecommerce.purchase_revenue,
  ecommerce.total_item_quantity

FROM
  `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
WHERE
  -- 取 2020 年 11 月 1 日至 11 月 30 日 (共 30 天)
  _TABLE_SUFFIX BETWEEN '20201101' AND '20201130'
  -- 只撈取購物漏斗相關事件，排除不必要的背景事件以節省空間
  AND event_name IN (
    'session_start',
    'view_item',
    'add_to_cart',
    'begin_checkout',
    'purchase'
  )


建立資料表 (Schema)

In [ ]:
CREATE SCHEMA IF NOT EXISTS raw;

CREATE TABLE raw.ga4_events (
    event_date INT,
    event_time TIMESTAMP,
    event_name VARCHAR(50),
    user_pseudo_id VARCHAR(100),
    session_id BIGINT,
    device_category VARCHAR(50),
    operating_system VARCHAR(50),
    country VARCHAR(100),
    city VARCHAR(100),
    traffic_source VARCHAR(100),
    traffic_medium VARCHAR(100),
    purchase_revenue NUMERIC,
    total_item_quantity INT
);


In [ ]:
-- 在 PostgreSQL 執行，加速 Power BI 的查詢
CREATE INDEX idx_fact_customer_id ON marts.fact_sessions(customer_id);
CREATE INDEX idx_fact_traffic_sk ON marts.fact_sessions(traffic_sk);


**問題(1)**

dbt 官方的最佳實踐是將 staging 模型設定為 view 。但在處理 GA4 日誌這類巨量且持續增加（Append-only）的時間序列資料時，將 stg_ga4_events.sql 改為 incremental 是解決效能瓶頸的常見做法 。

如果設定為 view，每次下游的 fact_sessions 或 dim_traffic 去查詢它時，這些 CASE WHEN 邏輯就會針對全部 150 萬筆資料重新運算一次 。

如果設定為 table，雖然下游查詢變快了，但每次 dbt run 時，你還是得把這 30 天的資料全部重新清理並重寫入硬碟。

改用 incremental 後，dbt 每天只會對「昨天剛匯入的新資料」執行 CASE WHEN 清理，並附加到實體表中，大幅節省運算資源 。



只需要用 event_time 作為過濾條件，就能輕鬆實現增量更新：

In [ ]:
{% if is_incremental() %}
  WHERE event_time > (SELECT max(event_time) FROM {{ this }})
{% endif %}
  

**問題(2)**

編寫dbt 時出現Windows 繁體中文系統的經典坑。

cp950（Big5 編碼）無法解讀 UTF-8 的中文字元，而 dbt 在解析 dbt_project.yml 時用了系統預設編碼。

**問題(3)**

PowerBI 圖表發現 is_begin_checkout 數量比 is_add_to_cart 多, 使用SQL 查詢驗證

In [ ]:
-- 在 PostgreSQL 執行這段 SQL 驗證 session 觸發了 begin_checkout 但完全沒有 add_to_cart 事件
SELECT COUNT(*) AS sessions_checkout_no_cart
FROM marts.fact_sessions
WHERE is_begin_checkout = 1
  AND is_add_to_cart = 0;

Output: 3532 


Note: begin_checkout count exceeds add_to_cart because GA4 tracking does not always capture the add_to_cart event — some users proceed directly to checkout via "Buy Now" or saved cart items